In [1]:
# Cell 1
# Mục đích: tạo kết nối và các hàm đo byte dùng chung (chỉ cần khi notebook mới).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")   # kết nối, vùng dữ liệu US

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi đọc job quá sớm lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng byte, không chạy thật, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache để số đo là số thật, có trần bytes billed
    cfg = bigquery.QueryJobConfig(use_query_cache=False, maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()            # chờ job xong rồi lấy kết quả
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [ ]:
# Cell 2
# Mục đích: kiểm tra fct_net_sales: đúng grain, không mất dòng khi join, khớp staging cùng lúc.
sql = f"""
select
    (select count(*) from `{PROJECT}.dbt_dev.fct_net_sales`)                          as fct_rows,
    (select count(distinct order_item_id) from `{PROJECT}.dbt_dev.fct_net_sales`)     as fct_distinct_ids,
    (select countif(cost is null) from `{PROJECT}.dbt_dev.fct_net_sales`)             as u,
    (select count(*) from `{PROJECT}.dbt_dev.stg_thelook__order_items`
      where status = 'Complete')                                                       as stg_complete_rows
"""
check(sql)                # xem byte trước
df = stats(sql)
print(len(df))            # phải là 1 dòng
df

Se quet: 2.47 MB
processed: 2.47 MB | billed: 20.0 MB
1


,fct_rows,fct_distinct_ids,fct_null_cost,stg_complete_rows
0,45529,45529,0,45529


In [3]:
# Cell 3
# Mục đích: so byte của cùng một phép tính trên view (staging) và trên table (mart).
sql_view = f"""
select count(*) as so_dong, sum(sale_price) as net_revenue
from `{PROJECT}.dbt_dev.stg_thelook__order_items`
where status = 'Complete'
"""
sql_table = f"""
select count(*) as so_dong, sum(sale_price) as net_revenue
from `{PROJECT}.dbt_dev.fct_net_sales`
"""
print("VIEW  (stg):")
df_v = stats(sql_view)
print("TABLE (fct):")
df_t = stats(sql_table)
print(df_v.equals(df_t))   # hai kết quả có giống nhau không

VIEW  (stg):
processed: 3.16 MB | billed: 10.0 MB
TABLE (fct):
processed: 0.35 MB | billed: 10.0 MB
True
